# Cache chữ RegionCLIP cho prompt đối tượng
Gắn checkpoint RegionCLIP RN50 đã chạy ở v135 và **csa-prompt-full-karpathy-v175**. Dùng 1 GPU T4 và bật Internet. Gửi **regionclip_object_text_v1_review.zip**.


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, zipfile
CHECKPOINT_PATH = ''  # Optional full path to the mounted official RN50 checkpoint.
FILENAMES = ('regionclip_pretrained-cc_rn50.pth', 'regionclip_pretrained-cc_rn50.bin')
REVISION = '4b8513b56e24827e3d6468e1f2105869f35c2d0b'
if CHECKPOINT_PATH:
    CHECKPOINT = Path(CHECKPOINT_PATH)
    if not CHECKPOINT.is_file() or CHECKPOINT.name not in FILENAMES:
        raise FileNotFoundError(f'Expected an RN50 checkpoint at {CHECKPOINT}')
else:
    candidates = sorted(path for name in FILENAMES for path in Path('/kaggle/input').rglob(name) if path.is_file())
    if not candidates:
        raise FileNotFoundError(f'Attach a Kaggle Dataset containing {FILENAMES[0]} or {FILENAMES[1]} via Add Input, then rerun this cell.')
    if len(candidates) > 1:
        raise ValueError('Set CHECKPOINT_PATH to one of these matches: '+', '.join(map(str,candidates)))
    CHECKPOINT = candidates[0]
print('RegionCLIP checkpoint:', CHECKPOINT, flush=True)
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
if not PROMPT_ROOT.is_dir():
    raise FileNotFoundError('Attach v175 prompt links or fix PROMPT_ROOT')
OUTPUT = Path('/kaggle/working/regionclip_object_text_v1')
OUTPUT.mkdir(exist_ok=False)
ROOT=Path(tempfile.mkdtemp(prefix='regionclip-fixed-roi-'))
REPO=ROOT/'RegionCLIP'
DEPS=ROOT/'deps'
env=dict(os.environ,MAX_JOBS='2',CUDA_VISIBLE_DEVICES='')
def run(args, timeout, environment=None, cwd=None):
    with (OUTPUT/'notebook.log').open('a',encoding='utf8') as log:
        p=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,encoding='utf8',errors='replace',env=environment,cwd=cwd)
        try:
            result, _=p.communicate(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.kill()
            result, _=p.communicate()
            log.write(result)
            raise
        log.write(result)
        print(result[-6000:])
        if p.returncode:
            raise subprocess.CalledProcessError(p.returncode,args)


In [ ]:
WORKER_SOURCE="from pathlib import Path\nimport hashlib,json,sys\nimport numpy as np\nimport torch\nfrom detectron2.config import get_cfg\nfrom detectron2.data.datasets.clip_prompt_utils import tokenize\nfrom detectron2.modeling.backbone.clip_backbone import build_clip_language_encoder\n\nrepo, checkpoint, out, prompt_root=map(Path,sys.argv[1:])\ndef sha(path):\n    with path.open('rb') as stream:\n        return hashlib.file_digest(stream,'sha256').hexdigest()\nexpected='afbc58d7d4bd96e2b10cc0d1928a912ed519c3b3215df7e5f3285e4bb09c7c58'\nif sha(checkpoint)!=expected:\n    raise ValueError('Expected the exact v135 RN50 source checkpoint')\nlinks,=prompt_root.rglob('prompt_links.jsonl')\nlink_sha=sha(links)\nif link_sha!='e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0':\n    raise ValueError('Unexpected v175 links')\nvocabulary=set()\nwith links.open(encoding='utf8') as stream:\n    for row in map(json.loads,stream):\n        if row['group'] in ('train','val'):\n            vocabulary.update(slot['text'] for slot in row['objects'])\ntexts=sorted(vocabulary)\nif not texts:\n    raise ValueError('Missing object prompt text')\ncfg=get_cfg()\ncfg.merge_from_file(str(repo/'configs/LVISv1-InstanceSegmentation/CLIP_fast_rcnn_R_50_C4_zsinf.yaml'))\nencoder=build_clip_language_encoder(cfg)\npayload=torch.load(checkpoint,map_location='cpu',weights_only=False)\nstate={key.removeprefix('module.'):value for key,value in payload['model'].items()}\nweights={key.removeprefix('lang_encoder.'):torch.as_tensor(value)\n         for key,value in state.items() if key.startswith('lang_encoder.')}\nmissing=sorted(set(encoder.state_dict())-set(weights))\nif missing:\n    (out/'checkpoint_inventory.json').write_text(json.dumps(dict(\n           checkpoint_sha256=expected, lang_keys=len(weights), missing=missing,\n           prefixes=sorted({key.split('.')[0] for key in state})),indent=2))\n    raise ValueError(f'RN50 checkpoint lacks {len(missing)} required language tensors; see inventory')\nencoder.load_state_dict(weights,strict=True)\ndel weights,state,payload\nif not torch.cuda.is_available():\n    raise RuntimeError('Select GPU T4')\ndevice=torch.device('cuda:0')\nencoder=encoder.to(device).eval()\nvectors=[]\nwith torch.inference_mode():\n    for i in range(0,len(texts),32):\n        tokens=tokenize(texts[i:i+32]).to(device)\n        result=encoder.encode_text(tokens).float()\n        if not torch.isfinite(result).all() or (result.norm(dim=-1)<=0).any():\n            raise ValueError('Invalid language embeddings')\n        vectors.append(torch.nn.functional.normalize(result,dim=-1).cpu().numpy())\nmatrix=np.concatenate(vectors)\nif matrix.shape!=(len(texts),1024):\n    raise ValueError('Wrong language embedding dimensions')\nnames=out/'object_texts.json'\nnames.write_text(json.dumps(texts,ensure_ascii=False,indent=2),encoding='utf8')\nvalues=out/'object_text_embeddings.npy'\nnp.save(values,matrix,allow_pickle=False)\n(out/'result.json').write_text(json.dumps(dict(checkpoint_sha256=expected,prompt_links_sha256=link_sha,\n    shape=list(matrix.shape),texts_sha256=sha(names),embeddings_sha256=sha(values)),indent=2))\n"
WORKER = ROOT/'run.py'
WORKER.write_text(WORKER_SOURCE,encoding='utf8')
try:
    run(['git','init','-q',REPO],30)
    run(['git','-C',REPO,'fetch','--depth','1','https://github.com/microsoft/RegionCLIP.git',REVISION],180)
    run(['git','-C',REPO,'checkout','--detach',REVISION],60)
    run([sys.executable,'-m','pip','install','--target',DEPS,
         'numpy>=1.26,<2.3','yacs','fvcore','iopath','termcolor','diffdist','ftfy','pycocotools','omegaconf'],240)
    setup_path=REPO/'setup.py'
    setup_source=setup_path.read_text(encoding='utf8')
    version_option='version=get_version(),'
    if setup_source.count(version_option)!=1:
        raise RuntimeError('RegionCLIP setup version field differs from the pinned source')
    setup_path.write_text(setup_source.replace(version_option,'version="0.0.0",'),encoding='utf8')
    transform_path=REPO/'detectron2/data/transforms/transform.py'
    transform_source=transform_path.read_text(encoding='utf8')
    old_default='interp=Image.LINEAR,'
    if transform_source.count(old_default)!=1:
        raise RuntimeError('RegionCLIP Pillow default differs from the pinned source')
    transform_path.write_text(transform_source.replace(
        old_default,'interp=Image.Resampling.BILINEAR,'),encoding='utf8')
    for module in ('rcnn.py','clip_rcnn.py'):
        module_path=REPO/'detectron2/modeling/meta_arch'/module
        module_source=module_path.read_text(encoding='utf8')
        old_import='from numpy.lib import pad'
        if module_source.count(old_import)!=1:
            raise RuntimeError(f'Unexpected NumPy import in {module}')
        module_path.write_text(module_source.replace(old_import,'from numpy import pad'),encoding='utf8')
    env['PYTHONPATH']=str(DEPS)
    run([sys.executable,'setup.py','build_ext','--inplace'],480,env,REPO)
    inference_env=dict(os.environ,PYTHONPATH=os.pathsep.join((str(REPO),str(DEPS))))
    run([sys.executable,WORKER,REPO,CHECKPOINT,OUTPUT,PROMPT_ROOT],1200,inference_env)
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('notebook.log','result.json','object_texts.json','object_text_embeddings.npy','checkpoint_inventory.json'):
            if (OUTPUT/name).exists():
                z.write(OUTPUT/name,name)
    print('SEND REVIEW',review)
